In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### Dataset Loading and Formatting Dataset

In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("yahma/alpaca-cleaned")
dataset

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})

In [4]:
dataset['train'][0]['output']

'1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [5]:
train_dataset = dataset['train']
train_dataset[10]

{'output': 'The capital city of France is Paris.',
 'input': '',
 'instruction': 'What is the capital of France?'}

In [6]:
train_dataset.shape

(51760, 3)

In [7]:
# Create Formatting Function

# in this func we remove the input section if the input is empty cause it costs us the extra tokens..

def formatting_func(example):
  instruction = example["instruction"].strip()
  input = example["input"].strip()
  output = example["output"].strip()

  if(input.strip()):
    formatted_text = f""" ### Instruction : {instruction}
    ### Input : {input}
    ### Response : {output}
    """
  else:
    formatted_text = f""" ### Instruction : {instruction}
    ### Response : {output}
    """
  return {"text":formatted_text}

In [8]:
# apply formatting func
formatted_dataset = train_dataset.map(formatting_func)
formatted_dataset

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 51760
})

In [9]:
formatted_dataset[0]['text']

' ### Instruction : Give three tips for staying healthy.\n    ### Response : 1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.\n    '

#### Tokenization of the Dataset

In [10]:
from transformers import AutoTokenizer , AutoModelForCausalLM

In [11]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [12]:

MAX_LENGTH = 512
def tokenize_func(example):
  return tokenizer(
      example["text"],
      padding=False,
      turcation=True,
      max_length=MAX_LENGTH,
      return_attention_mask=True,
      return_token_type_ids=True
  )

In [13]:
tokenized_dataset = formatted_dataset.map(tokenize_func,
                                          batched=True,
                                          remove_columns=['input','instruction','output','text'],
                                          )

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [14]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 51760
})

In [15]:
tokenized_dataset[0]['attention_mask']

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [16]:
# for the base model the "token_type_ids" doesn't required
tokenized_dataset = tokenized_dataset.remove_columns(['token_type_ids'])
tokenized_dataset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 51760
})

#### Parameter Efficient Fine Tuning (PEFT) / LoRA

In [17]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [18]:
from peft import LoraConfig
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)

In [19]:
!pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [20]:
from peft import get_peft_model
peft_model = get_peft_model(
    model,
    lora_config
)
peft_model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


#### Define the Training Arguments

In [42]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./qwen-alpaca-lora',
    # training
    per_device_train_batch_size = 4,
    num_train_epochs=2,
    gradient_accumulation_steps=4,
    # Optimization
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    # logging
    logging_steps=100,
    #saving
    save_strategy='epoch',
    save_total_limit=2,

    fp16=True,
    seed=42
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [43]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [44]:
from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

#### Training the peft Model

In [45]:
trainer.train()

Step,Training Loss
100,1.370105
200,1.330875
300,1.321971
400,1.323195
500,1.311235
600,1.312444
700,1.362094
800,1.365875
900,1.367163
1000,1.354904


TrainOutput(global_step=6470, training_loss=1.3335697521934922, metrics={'train_runtime': 8324.4505, 'train_samples_per_second': 12.436, 'train_steps_per_second': 0.777, 'total_flos': 6.87384612820224e+16, 'train_loss': 1.3335697521934922, 'epoch': 2.0})

In [46]:
trainer.save_model("./qwen-alpaca-lora")

In [48]:
tokenizer.save_pretrained("./qwen-alpaca-lora")

('./qwen-alpaca-lora/tokenizer_config.json',
 './qwen-alpaca-lora/chat_template.jinja',
 './qwen-alpaca-lora/tokenizer.json')

In [50]:
#  reload the base model
from transformers import AutoTokenizer , AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [52]:
# load LoRA Adapter
from peft import PeftModel
model = PeftModel.from_pretrained(model,'./qwen-alpaca-lora')
model.eval()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): Qwen2ForCausalLM(
          (model): Qwen2Model(
            (embed_tokens): Embedding(151936, 896)
            (layers): ModuleList(
              (0-23): 24 x Qwen2DecoderLayer(
                (self_attn): Qwen2Attention(
                  (q_proj): lora.Linear(
                    (base_layer): Linear(in_features=896, out_features=896, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=896, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=896, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
                    (lora

In [98]:
prompt = """ " Implement Dijkstra's shortest path algorithm using a priority queue in C++."

"""

In [99]:
# tokenize the prompt
inputs=tokenizer(prompt,return_tensors='pt')

In [106]:
import torch
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

In [107]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

 " Implement Dijkstra's shortest path algorithm using a priority queue in C++."

// Problem:
// Given an adjacency matrix of a weighted graph, implement Dijkstra's shortest path algorithm to find the shortest paths from a starting vertex to all other vertices.
// The function should return a vector containing the shortest distances from the starting vertex to each vertex.

#include <iostream>
#include <vector>
#include <queue>
using namespace std;

class Graph {
private:
    int V; // Number of vertices
    vector<vector<int>> adj; // Adjacency list

public:
    Graph(int V) : V(V), adj(V) {}

    void addEdge(int v, int w, int cost) { // Add edge with given weight
        adj[v].push_back(w); // Add neighbor
        adj[w].push_back(v); // Add neighbor
        adj[v][w] = cost; // Weighted edge
        adj[w][v] = cost; // Weighted edge
    }

    // Helper function to print the path and distance
    void printPath(vector<int>& path, int src, int dest) {
        while (dest != -1) {
 

In [109]:
results = train_dataset.filter(
    lambda x: " Implement Dijkstra's shortest path algorithm using a priority queue in C++." in x["instruction"].lower()
)

print(len(results))

Filter:   0%|          | 0/51760 [00:00<?, ? examples/s]

0


In [111]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Reload the base model to ensure it's a clean AutoModelForCausalLM instance
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
base_model_for_merge = AutoModelForCausalLM.from_pretrained(model_name)

# Load the LoRA adapter onto this fresh base model
peft_model_for_merge = PeftModel.from_pretrained(base_model_for_merge, './qwen-alpaca-lora')

# Now, merge and unload the correctly structured PEFT model
merged_model = peft_model_for_merge.merge_and_unload()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [112]:
merged_model.save_pretrained('./merged_model')
tokenizer.save_pretrained('./merged_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./merged_model/tokenizer_config.json',
 './merged_model/chat_template.jinja',
 './merged_model/tokenizer.json')